In [1]:
import nltk
from nltk.corpus import brown
from collections import defaultdict
import random

# Download required NLTK data
nltk.download('brown')
nltk.download('universal_tagset')

[nltk_data] Downloading package brown to /root/nltk_data...
[nltk_data]   Unzipping corpora/brown.zip.
[nltk_data] Downloading package universal_tagset to /root/nltk_data...
[nltk_data]   Unzipping taggers/universal_tagset.zip.


True

STEP 1: Load and preprocess data

In [2]:
sentences = brown.tagged_sents(tagset='universal')
print(f"Loaded {len(sentences)} sentences")

processed_sentences = []

def clean_word(word):
    word = word.lower()
    # Keep only alphabetic words (no punctuation, no numbers)
    if word.isalpha():
        return word
    return None

for sent in sentences:
    cleaned = []
    for word, tag in sent:
        w = clean_word(word)
        if w:
            cleaned.append((w, tag))
    # Only keep sentences with at least 2 words after cleaning
    if len(cleaned) > 1:
        processed_sentences.append(cleaned)

print(f"After preprocessing: {len(processed_sentences)} sentences")

Loaded 57340 sentences
After preprocessing: 55820 sentences


STEP 2: Build states (POS tags) and vocabulary

In [3]:
states = set()
vocab = set()

for sent in processed_sentences:
    for word, tag in sent:
        states.add(tag)
        vocab.add(word)

states = sorted(list(states))
vocab = sorted(list(vocab))

print(f"Number of states (POS tags): {len(states)}")
print(f"Vocabulary size: {len(vocab)}")
print(f"States: {states}")

Number of states (POS tags): 11
Vocabulary size: 40189
States: ['ADJ', 'ADP', 'ADV', 'CONJ', 'DET', 'NOUN', 'NUM', 'PRON', 'PRT', 'VERB', 'X']


STEP 3: Build transition counts (tag → next tag)

In [4]:
transition_counts = defaultdict(lambda: defaultdict(int))
tag_counts = defaultdict(int)

for sent in processed_sentences:
    for i in range(len(sent) - 1):
        tag1 = sent[i][1]
        tag2 = sent[i+1][1]
        transition_counts[tag1][tag2] += 1
        tag_counts[tag1] += 1

STEP 4: Build emission counts (tag → word)

In [5]:
emission_counts = defaultdict(lambda: defaultdict(int))
tag_word_counts = defaultdict(int)

for sent in processed_sentences:
    for word, tag in sent:
        emission_counts[tag][word] += 1
        tag_word_counts[tag] += 1

STEP 5: Add Laplace smoothing and convert to probabilities

In [6]:
alpha = 0.1  # Small smoothing factor

# Transition probabilities with smoothing
transition_prob = defaultdict(dict)
for tag1 in transition_counts:
    total_transitions = sum(transition_counts[tag1].values())
    for tag2 in states:
        count = transition_counts[tag1][tag2]
        transition_prob[tag1][tag2] = (count + alpha) / (total_transitions + alpha * len(states))

# Emission probabilities with smoothing
emission_prob = defaultdict(dict)
for tag in emission_counts:
    total_words = sum(emission_counts[tag].values())
    for word in vocab:
        count = emission_counts[tag][word]
        emission_prob[tag][word] = (count + alpha) / (total_words + alpha * len(vocab))

STEP 6: Build word-to-tags index

In [7]:
word_to_tags = defaultdict(set)
for sent in processed_sentences:
    for word, tag in sent:
        word_to_tags[word].add(tag)

print(f"Built index for {len(word_to_tags)} words")

Built index for 40189 words


STEP 7: Prediction function

In [8]:
def predict_next_word(current_word, top_k=5, avoid_common=False):
    """
    Predict the next word using HMM probabilities

    Parameters:
    - current_word: input word
    - top_k: consider top k most likely words for each tag
    - avoid_common: if True, skip very common stopwords in predictions
    """
    current_word = current_word.lower()

    # Common stopwords to avoid (if specified)
    common_stopwords = {'the', 'a', 'an', 'and', 'or', 'but', 'in', 'on', 'at', 'to', 'for', 'of', 'with', 'by'}

    # Handle unknown words by using most common starting words
    if current_word not in word_to_tags:
        print(f"  Note: '{current_word}' not in vocabulary, using fallback")
        # Return a contextually appropriate word based on POS patterns
        return random.choice(['the', 'a', 'is', 'was', 'in'])

    best_candidates = []

    # For each possible tag of the current word
    for current_tag in word_to_tags[current_word]:
        if current_tag not in transition_prob:
            continue

        # Find the most likely next tags
        next_tags = sorted(transition_prob[current_tag].items(),
                          key=lambda x: x[1], reverse=True)[:top_k]

        for next_tag, trans_prob in next_tags:
            if next_tag not in emission_prob:
                continue

            # Get most likely words for this next tag
            word_probs = []
            for word, emit_prob in emission_prob[next_tag].items():
                # Skip if we're avoiding common stopwords
                if avoid_common and word in common_stopwords:
                    continue
                score = trans_prob * emit_prob
                word_probs.append((word, score))

            # Sort by probability and take top k
            word_probs.sort(key=lambda x: x[1], reverse=True)
            best_candidates.extend(word_probs[:top_k])

    # If no candidates found, return a fallback
    if not best_candidates:
        return random.choice(['the', 'is', 'was', 'of', 'in'])

    # Sort all candidates by score
    best_candidates.sort(key=lambda x: x[1], reverse=True)

    # Return the top candidate
    return best_candidates[0][0]

STEP 8: Test the model

In [9]:
test_words = ['the', 'time', 'india', 'ball', 'tennis', 'hello', 'computer', 'man', 'woman']

print("Predictions with basic mode:")
for word in test_words:
    prediction = predict_next_word(word, top_k=3, avoid_common=False)
    print(f"  '{word}' -> '{prediction}'")

print("\nPredictions avoiding common stopwords:")
for word in test_words:
    prediction = predict_next_word(word, top_k=3, avoid_common=True)
    print(f"  '{word}' -> '{prediction}'")

print("\n=== STEP 9: Interactive mode ===")
print("Enter words to get predictions (type 'quit' to exit)")
print("=" * 50)

while True:
    user_input = input("\nEnter a word: ").strip()

    if user_input.lower() == 'quit':
        print("Goodbye!")
        break

    if not user_input:
        continue

    # Get predictions with different settings
    print(f"\nPredictions for '{user_input}':")

    # Basic prediction
    pred1 = predict_next_word(user_input, top_k=3, avoid_common=False)
    print(f"  Basic: '{pred1}'")

    # Prediction avoiding stopwords
    pred2 = predict_next_word(user_input, top_k=3, avoid_common=True)
    if pred2 != pred1:
        print(f"  (Avoiding stopwords): '{pred2}'")

    # Show multiple candidates
    print(f"\n  Top 5 candidate words:")
    current_word = user_input.lower()
    all_candidates = []

    if current_word in word_to_tags:
        for current_tag in word_to_tags[current_word]:
            if current_tag in transition_prob:
                next_tags = sorted(transition_prob[current_tag].items(),
                                  key=lambda x: x[1], reverse=True)[:3]
                for next_tag, trans_prob in next_tags:
                    if next_tag in emission_prob:
                        # Get top words for this tag
                        for word, emit_prob in emission_prob[next_tag].items():
                            score = trans_prob * emit_prob
                            all_candidates.append((word, score, next_tag))

    # Sort and show top candidates
    all_candidates.sort(key=lambda x: x[1], reverse=True)
    for i, (word, score, tag) in enumerate(all_candidates[:5]):
        print(f"    {i+1}. '{word}' (tag: {tag}, score: {score:.6f})")


Predictions with basic mode:
  'the' -> 'de'
  'time' -> 'the'
  'india' -> 'of'
  'ball' -> 'of'
  'tennis' -> 'of'
  'hello' -> 'the'
  'computer' -> 'of'
  'man' -> 'the'
  'woman' -> 'the'

Predictions avoiding common stopwords:
  'the' -> 'de'
  'time' -> 'that'
  'india' -> 'that'
  'ball' -> 'that'
  'tennis' -> 'that'
  'hello' -> 'is'
  'computer' -> 'that'
  'man' -> 'is'
  'woman' -> 'that'

=== STEP 9: Interactive mode ===
Enter words to get predictions (type 'quit' to exit)

Enter a word: man

Predictions for 'man':
  Basic: 'the'
  (Avoiding stopwords): 'is'

  Top 5 candidate words:
    1. 'the' (tag: DET, score: 0.087751)
    2. 'of' (tag: ADP, score: 0.078615)
    3. 'of' (tag: ADP, score: 0.045784)
    4. 'in' (tag: ADP, score: 0.045053)
    5. 'the' (tag: DET, score: 0.044057)

Enter a word: quit
Goodbye!
